# Telegram Summary Pipeline

Capital IQ transcript 요약을 Grok으로 생성하고 Telegram 전송용 형식으로 확인하는 notebook.


In [1]:
from __future__ import annotations

import csv
import html
import importlib.util
import json
import os
import re
import time
from datetime import datetime
from pathlib import Path

import requests


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "scripts" / "collect_capiq_transcripts.py").exists():
            return candidate
    raise FileNotFoundError("Could not find project root from current working directory.")


def norm(value: str) -> str:
    return re.sub(r"\s+", " ", value or "").strip()


def slugify(value: str) -> str:
    value = re.sub(r"[<>:\"/\\|?*]", " ", value or "")
    value = re.sub(r"\s+", "_", value).strip("._ ")
    return value or "unknown"


def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


REQUIRED_PACKAGES = {
    "requests": "requests",
    "docx": "python-docx",
    "pypdf": "pypdf",
}
missing_packages = [package_name for module_name, package_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module_name) is None]
if missing_packages:
    print("Missing packages in the active Jupyter kernel:")
    for package_name in missing_packages:
        print(f"- {package_name}")
    print()
    print("Run this in a notebook cell and then rerun Cell 1:")
    print(f"%pip install {' '.join(missing_packages)}")
else:
    print("All required packages are available in the active Jupyter kernel.")


def read_docx_text(path: Path) -> str:
    try:
        from docx import Document
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError("python-docx is not installed in the active Jupyter kernel. Run: %pip install python-docx") from exc
    document = Document(path)
    lines = [norm(paragraph.text) for paragraph in document.paragraphs]
    return "\n".join(line for line in lines if line)


def read_pdf_text(path: Path) -> str:
    try:
        from pypdf import PdfReader
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError("pypdf is not installed in the active Jupyter kernel. Run: %pip install pypdf") from exc
    reader = PdfReader(str(path))
    parts = []
    for page in reader.pages:
        parts.append(norm(page.extract_text() or ""))
    return "\n".join(part for part in parts if part)


def load_transcript_text(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix == ".docx":
        return read_docx_text(path)
    if suffix == ".pdf":
        return read_pdf_text(path)
    raise ValueError(f"Unsupported transcript format: {path.suffix}")


def split_text(text: str, max_chars: int) -> list[str]:
    text = norm(text)
    if len(text) <= max_chars:
        return [text]
    chunks = []
    cursor = 0
    while cursor < len(text):
        end = min(cursor + max_chars, len(text))
        if end < len(text):
            split_at = text.rfind(" ", cursor, end)
            if split_at > cursor + max_chars // 2:
                end = split_at
        chunks.append(text[cursor:end].strip())
        cursor = end
    return [chunk for chunk in chunks if chunk]


def extract_response_text(data: dict) -> str:
    texts = []
    for item in data.get("output", []):
        if item.get("type") != "message":
            continue
        for content in item.get("content", []):
            if content.get("type") in {"output_text", "text"}:
                text = content.get("text", "")
                if text:
                    texts.append(text)
    return "\n".join(texts).strip()


def call_grok(*, api_key: str, model: str, system_prompt: str, user_prompt: str, max_output_tokens: int, base_url: str, store: bool, timeout_sec: int) -> str:
    response = requests.post(
        f"{base_url}/responses",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": model,
            "input": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            "max_output_tokens": max_output_tokens,
            "store": store,
        },
        timeout=timeout_sec,
    )
    response.raise_for_status()
    data = response.json()
    text = extract_response_text(data)
    if not text:
        raise RuntimeError(f"No text output found in Grok response: {json.dumps(data, ensure_ascii=False)[:1000]}")
    return text


def choose_transcript_files(input_dir: Path, format_preference: list[str]) -> list[Path]:
    preference_rank = {fmt.lower(): i for i, fmt in enumerate(format_preference)}
    candidates = [path for path in input_dir.iterdir() if path.is_file() and path.suffix.lower() in {'.docx', '.pdf'}]
    grouped = {}
    for path in candidates:
        stem = path.stem.casefold()
        grouped.setdefault(stem, []).append(path)

    selected = []
    for paths in grouped.values():
        selected.append(
            sorted(
                paths,
                key=lambda p: (preference_rank.get(p.suffix.lower().lstrip('.'), 99), p.name.casefold()),
            )[0]
        )
    return sorted(selected, key=lambda p: p.name.casefold())


def extract_ticker(text: str) -> str:
    match = re.search(r"\b[A-Z][A-Za-z]+[: ]([A-Z]{1,6})\b", text)
    return match.group(1) if match else "확인 불가"


def extract_quarter(text: str) -> str:
    match = re.search(r"\b(FQ\d\s+\d{4}|Q\d\s+\d{4})\b", text)
    return match.group(1) if match else "확인 불가"


def extract_session(text: str) -> str:
    for line in text.splitlines()[:20]:
        lowered = line.lower()
        if "earnings call" in lowered or "conference call" in lowered:
            return norm(line)
    return "확인 불가"


def extract_metadata_block(path: Path, transcript_text: str) -> str:
    lines = [norm(line) for line in transcript_text.splitlines() if norm(line)]
    top_lines = lines[:25]
    consensus_lines = []
    capture = False
    for line in top_lines:
        if "CONSENSUS" in line.upper() and "ACTUAL" in line.upper():
            capture = True
            consensus_lines.append(line)
            continue
        if capture:
            if len(consensus_lines) >= 8:
                break
            consensus_lines.append(line)
            if line.lower().startswith("currency:"):
                break

    metadata_lines = [
        f"File Name: {path.name}",
        f"Source Path: {path}",
        f"Ticker: {extract_ticker(transcript_text)}",
        f"Quarter: {extract_quarter(transcript_text)}",
        f"Session: {extract_session(transcript_text)}",
        "Top Header Lines:",
    ]
    metadata_lines.extend(f"- {line}" for line in top_lines[:10])
    if consensus_lines:
        metadata_lines.append("Consensus / Actual / Surprise Block:")
        metadata_lines.extend(f"- {line}" for line in consensus_lines)
    else:
        metadata_lines.append("Consensus / Actual / Surprise Block: 확인 불가")
    return "\n".join(metadata_lines)


PROJECT_ROOT = find_project_root(Path.cwd())
ENV_PATH = PROJECT_ROOT / ".env"
load_env_file(ENV_PATH)
SUMMARY_ROOT = PROJECT_ROOT / "output" / "summaries" / "grok"
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Loaded .env: {ENV_PATH.exists()}")
print(f"SUMMARY_ROOT: {SUMMARY_ROOT}")


All required packages are available in the active Jupyter kernel.
PROJECT_ROOT: E:\Earnings
Loaded .env: True
SUMMARY_ROOT: E:\Earnings\output\summaries\grok


## 2. 통신 확인

프롬프트 입력 전에 가장 단순한 형태로 Grok API와 통신이 되는지만 확인한다.


In [2]:
XAI_API_KEY = os.getenv("XAI_API_KEY", "").strip()
XAI_MODEL = os.getenv("XAI_MODEL", "grok-4.5").strip() or "grok-4.5"
XAI_BASE_URL = os.getenv("XAI_BASE_URL", "https://api.x.ai/v1").rstrip("/")
STORE_REMOTE_HISTORY = False
REQUEST_TIMEOUT_SEC = 600

if not XAI_API_KEY:
    raise RuntimeError("Set XAI_API_KEY in your environment first.")

connection_check = call_grok(
    api_key=XAI_API_KEY,
    model=XAI_MODEL,
    system_prompt="You are a network connectivity checker.",
    user_prompt="Return exactly: CONNECTION_OK",
    max_output_tokens=20,
    base_url=XAI_BASE_URL,
    store=STORE_REMOTE_HISTORY,
    timeout_sec=REQUEST_TIMEOUT_SEC,
)

print("Connection check response:")
print(connection_check)


Connection check response:
CONNECTION_OK


## 3. Grok API Key 확인

이 셀은 실제로 Grok API를 한 번 더 호출해서 현재 API Key가 사용 가능한지 확인한다.


In [3]:
api_check_result = call_grok(
    api_key=XAI_API_KEY,
    model=XAI_MODEL,
    system_prompt="You are a concise API connectivity checker.",
    user_prompt="Reply with exactly: GROK_API_OK",
    max_output_tokens=20,
    base_url=XAI_BASE_URL,
    store=STORE_REMOTE_HISTORY,
    timeout_sec=REQUEST_TIMEOUT_SEC,
)

print("API check response:")
print(api_check_result)


API check response:
GROK_API_OK


## 4. 프롬프트 입력

아래 프롬프트를 직접 수정해서 요약 스타일을 바꿀 수 있다.


In [4]:
CHUNK_SYSTEM_PROMPT = """You are an equity research analyst extracting facts from one chunk of an earnings call transcript.

Use only the chunk and provided metadata.
Do not use external knowledge or fill gaps.

Extract only:
- Actual results and KPIs
- Consensus / actual / surprise comparisons when explicitly shown
- Guidance and changes in guidance or tone
- Important Q&A topics
- Risks, unresolved issues, and cases where management avoids quantification

Rules:
- Preserve figures, periods, currencies, and units.
- Distinguish actuals, guidance, targets, and expectations.
- If information is unclear or missing, say so.
- Output concise Korean bullet points only.
- Do not translate mechanically; rewrite into natural Korean analyst language.
- No investment recommendation."""

FINAL_SYSTEM_PROMPT = """You are an equity research analyst.

Analyze the transcript using only the provided transcript and metadata.
Do not use external knowledge or invent missing information.

Focus on:
1. Actual results and key KPIs
2. Material surprises with a valid comparison point
3. Guidance changes and management tone
4. The most investment-relevant Q&A topics
5. Key risks and unresolved issues

Rules:
- Distinguish actuals, guidance, targets, and expectations.
- Preserve original figures, periods, currencies, and units.
- Do not call something a surprise without a valid comparison point.
- Flag unclear, conflicting, undisclosed, or partially answered items.
- Write concise, professional Korean that sounds like a Korean equity analyst, not a translated transcript.
- Prefer short Telegram-friendly blocks: compact bullets, short paragraphs, and no Markdown tables.
- Avoid duplicated numbers across sections unless repetition is necessary for context.
- Write primarily in Korean. Use English terms selectively only when they are standard in Korean equity research or clearer than a forced Korean translation.
- Do not overuse English. Prefer natural Korean when the Korean term is standard and readable.
- Avoid awkward literal translations; choose the wording a Korean equity analyst would naturally use.
- No investment recommendation."""

FINAL_USER_PROMPT_TEMPLATE = """아래 earnings call transcript와 metadata만 바탕으로 실적발표 요약을 작성해줘.
외부 지식, 외부 검색, 추측은 사용하지 마라.
출력은 Telegram에서 바로 읽기 좋은 형식으로 작성한다. 번역투를 피하고, 한국어 리서치 코멘트처럼 자연스럽게 쓴다.
기본은 한국어로 쓰되, 한국어 리서치에서 관용적으로 쓰이는 영어 용어는 필요한 경우에만 제한적으로 사용한다.
영어를 남발하지 말고, 자연스러운 한국어 표현이 있으면 한국어를 우선한다.
어색한 직역 표현은 피하고, 한국 애널리스트가 실제 코멘트에서 쓸 법한 표현을 선택한다.
전체 출력은 Telegram 한 메시지에 들어갈 수 있도록 공백 포함 3,200자 안팎을 권장한다. 다만 중요한 투자 판단 근거는 누락하지 않는다.
각 bullet은 모바일에서 읽기 쉽게 간결하게 쓴다. 중복 숫자와 장황한 설명은 줄인다.
각 섹션 제목과 Q&A 질문 제목은 Telegram HTML bold 태그로 감싼다. 예: <b>📊 핵심 실적 vs 컨센서스</b>, <b>Q1. 북미 수요 회복</b>
Markdown table은 절대 사용하지 않는다. 표가 필요한 내용도 bullet로 풀어쓴다.
HTML 태그는 섹션 제목과 Q&A 질문 제목의 <b>...</b>만 사용한다.

<b>{{TICKER}} | {{SESSION}}</b>

<b>📊 핵심 실적 vs 컨센서스</b>
컨센서스 비교가 가능한 핵심 실적만 아래 형식으로 작성:
- EPS: {{actual}} vs 컨센 {{consensus}} ({{beat/miss/flat}} {{차이}})
- 매출: {{actual}} vs 컨센 {{consensus}} ({{beat/miss/flat}} {{차이}})
- 영업이익: {{actual}} vs 컨센 {{consensus}} ({{beat/miss/flat}} {{차이}})
영업이익, EBITDA, margin 등은 actual과 consensus가 모두 있을 때만 같은 형식으로 추가한다.

규칙:
- Metadata에 있을 때만 consensus 사용
- Actual과 Consensus가 모두 있을 때만 beat/miss 판단
- YoY / QoQ는 transcript에 있을 때만 사용
- GAAP / Non-GAAP 구분
- 컨센서스 비교가 불가능한 사업부/볼륨/마진/가격/채널 내용은 이 섹션에 쓰지 말고 아래 '특이사항 & 핵심 KPI'에 쓴다
- 컨센서스가 없는 항목은 이 섹션에서 생략한다

<b>💬 경영진 핵심 메시지</b>
2~3문장, 한 문단으로 요약:
- 전체 실적 흐름
- 강했던 사업부·지역·제품
- 약했던 사업부·지역·제품
- 수요, 가격, 볼륨, 믹스, 마진
- 다음 분기 또는 하반기 핵심 변수

<b>⚡ 서프라이즈 포인트</b>
2~3개, 형식: - 짧은 제목: 설명
비교 기준이 없으면 surprise라고 쓰지 마라.

<b>🧾 특이사항 & 핵심 KPI</b>
중요 수치 3~5개. 기간과 단위를 함께 쓸 것.

<b>🧭 가이던스 및 변경사항</b>
- transcript에 실질적인 guidance, outlook, tone change가 있을 때만 이 섹션을 출력한다.
- 정보가 없으면 이 섹션 전체를 생략한다.
- 의미 있는 항목만 bullet로 쓰고, 미제시 항목을 억지로 채우지 않는다.
- Markdown table은 사용하지 않는다.

변경은 다음 중 하나만 사용:
상향 / 하향 / 재확인 / 범위 축소 / 범위 확대 / 철회 / 정성적 변경 / 확인 불가

표 아래에는 필요한 경우에만 간단히 요약:
- 기존 대비 달라진 점 / 주요 전제 / 압박·상쇄 요인을 최대 2줄로 압축

<b>❓ Q&A 요약</b>
투자 중요도가 높은 질문을 최대 5개 선택하되, 각 Q&A는 핵심만 담아 2~3줄로 정리한다. 핵심 실적/KPI 섹션의 숫자를 반복하지 말고, 새 투자 판단 정보를 우선한다:
- 가이던스 달성 전제, 상·하방 요인, 경영진 확신도
- 수요의 질, 회복 시점, 채널·지역·제품별 변화
- 마진·비용·경쟁·규제 등 핵심 리스크와 실행 과제
- 수치는 Q&A에서 새로 공개되거나 해석상 중요한 경우에만 포함

<b>Q1. {{질문 주제}}</b>
- 질문:
- 답변 핵심:
- 투자 관점:

동일 우려가 반복되면 마지막에 추가:
<b>반복적으로 제기된 핵심 우려</b>
- {{주제}}: {{경영진 답변과 남은 불확실성}}

<b>⚠️ 핵심 리스크</b>
Transcript에서 확인되는 리스크 2~3개. 각 항목은 한 줄로 쓴다.

추가 규칙:
- Transcript와 Metadata에 없는 숫자나 내용을 만들지 않는다.
- Actual, guidance, target, expectation을 구분한다.
- percentage와 percentage point를 구분한다.
- 부정적 내용과 미해결 이슈를 누락하지 않는다.
- 별도 서론 없이 바로 출력한다.

[METADATA]
{{METADATA}}

[TRANSCRIPT]
{{TRANSCRIPT}}"""

CHUNK_CHAR_LIMIT = 18000
CHUNK_SUMMARY_MAX_OUTPUT_TOKENS = 800
FINAL_SUMMARY_MAX_OUTPUT_TOKENS = 2200
REQUEST_PAUSE_SEC = 1.0


## 5. 입력 폴더 지정

여기서 transcript 폴더를 지정한다. 현재 샘플 기준으로는 DOCX 상단에 consensus / actual / surprise가 잘 들어 있으므로 기본은 DOCX 우선이다. 나중에 PDF가 더 안정적이면 순서만 바꾸면 된다.


In [5]:
INPUT_TRANSCRIPT_DIR = PROJECT_ROOT / "transcripts" / "2026-07-10"
MAX_FILES = None  # 예: 1, 3. 전체면 None
FILE_FORMAT_PREFERENCE = ["docx", "pdf"]  # PDF 우선으로 바꾸려면 ["pdf", "docx"]

if not INPUT_TRANSCRIPT_DIR.exists():
    raise FileNotFoundError(f"Input transcript directory not found: {INPUT_TRANSCRIPT_DIR}")

selected_files = choose_transcript_files(INPUT_TRANSCRIPT_DIR, FILE_FORMAT_PREFERENCE)
if MAX_FILES:
    selected_files = selected_files[:MAX_FILES]

print(f"Input directory: {INPUT_TRANSCRIPT_DIR}")
print(f"Selected files: {len(selected_files)}")
print(f"Format preference: {FILE_FORMAT_PREFERENCE}")
for path in selected_files:
    print(f"- {path.name}")

if not selected_files:
    raise RuntimeError("No transcript files found in the input directory.")


Input directory: E:\Earnings\transcripts\2026-07-10
Selected files: 1
Format preference: ['docx', 'pdf']
- PepsiCo Inc._Earnings Call_2026-07-09_English.docx


## 6. 결과 확인

이 셀을 실행하면 지정한 폴더의 transcript를 요약하고, 결과를 저장한 뒤 첫 번째 요약문을 바로 보여준다.


In [6]:
results = []
output_dir = SUMMARY_ROOT / INPUT_TRANSCRIPT_DIR.name
output_dir.mkdir(parents=True, exist_ok=True)

for idx, path in enumerate(selected_files, start=1):
    print(f"[{idx}/{len(selected_files)}] Summarizing {path.name}")
    transcript_text = load_transcript_text(path)
    metadata_block = extract_metadata_block(path.relative_to(PROJECT_ROOT), transcript_text)
    ticker = extract_ticker(transcript_text)
    quarter = extract_quarter(transcript_text)
    session = extract_session(transcript_text)
    chunks = split_text(transcript_text, max_chars=CHUNK_CHAR_LIMIT)
    chunk_summaries = []

    for chunk_idx, chunk in enumerate(chunks, start=1):
        chunk_prompt = "[METADATA]\n" + metadata_block + "\n\n[CHUNK]\n" + chunk
        chunk_summary = call_grok(
            api_key=XAI_API_KEY,
            model=XAI_MODEL,
            system_prompt=CHUNK_SYSTEM_PROMPT,
            user_prompt=chunk_prompt,
            max_output_tokens=CHUNK_SUMMARY_MAX_OUTPUT_TOKENS,
            base_url=XAI_BASE_URL,
            store=STORE_REMOTE_HISTORY,
            timeout_sec=REQUEST_TIMEOUT_SEC,
        )
        chunk_summaries.append(f"### Chunk {chunk_idx}\n{chunk_summary}")
        time.sleep(REQUEST_PAUSE_SEC)

    final_prompt = FINAL_USER_PROMPT_TEMPLATE
    final_prompt = final_prompt.replace("{{TICKER}}", ticker)
    final_prompt = final_prompt.replace("{{QUARTER}}", quarter)
    final_prompt = final_prompt.replace("{{SESSION}}", session)
    final_prompt = final_prompt.replace("{{METADATA}}", metadata_block)
    final_prompt = final_prompt.replace("{{TRANSCRIPT}}", "\n\n".join(chunk_summaries))

    final_summary = call_grok(
        api_key=XAI_API_KEY,
        model=XAI_MODEL,
        system_prompt=FINAL_SYSTEM_PROMPT,
        user_prompt=final_prompt,
        max_output_tokens=FINAL_SUMMARY_MAX_OUTPUT_TOKENS,
        base_url=XAI_BASE_URL,
        store=STORE_REMOTE_HISTORY,
        timeout_sec=REQUEST_TIMEOUT_SEC,
    )

    result = {
        "file_name": path.name,
        "source_file": str(path.relative_to(PROJECT_ROOT)),
        "ticker": ticker,
        "quarter": quarter,
        "session": session,
        "model": XAI_MODEL,
        "source_characters": len(transcript_text),
        "chunk_count": len(chunks),
        "summary": final_summary,
        "summarized_at": datetime.now().isoformat(),
    }
    md_path = output_dir / f"{slugify(path.stem)}.md"
    md_path.write_text(final_summary, encoding="utf-8")
    result["summary_file"] = str(md_path.relative_to(PROJECT_ROOT))
    results.append(result)
    time.sleep(REQUEST_PAUSE_SEC)

batch_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
batch_json = output_dir / f"summary_batch_{batch_stamp}.json"
batch_csv = output_dir / f"summary_batch_{batch_stamp}.csv"
batch_json.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

with open(batch_csv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["file_name", "source_file", "ticker", "quarter", "session", "model", "source_characters", "chunk_count", "summary_file", "summarized_at"],
    )
    writer.writeheader()
    for row in results:
        writer.writerow({field: row.get(field, "") for field in writer.fieldnames})

print(f"Saved JSON: {batch_json}")
print(f"Saved CSV: {batch_csv}")
print()
print("First summary preview:")
print(results[0]["summary"])


[1/1] Summarizing PepsiCo Inc._Earnings Call_2026-07-09_English.docx
Saved JSON: E:\Earnings\output\summaries\grok\2026-07-10\summary_batch_20260710_153443.json
Saved CSV: E:\Earnings\output\summaries\grok\2026-07-10\summary_batch_20260710_153443.csv

First summary preview:
<b>PEP | FQ2 2026 Earnings Call</b>

<b>📊 핵심 실적 vs 컨센서스</b>
- EPS (Normalized): $2.20 vs 컨센 $2.21 (miss -0.45%)
- 매출: $24,181mm vs 컨센 $23,966.26mm (beat +0.90%)
- 통화: USD

<b>💬 경영진 핵심 메시지</b>
상반기 매출 약 7% 성장, 글로벌 볼륨은 식품 3%·음료 2%로 2022년 이후 최고 속도. 성장 축은 국제(7% 가속)이며 미국 식품은 볼륨·점유율 회복에 성공. 다만 북미는 가스 가격·impulse 채널 약세로 Q2 기대 미달. 하반기는 국제 강세 지속, 북미 점진 개선을 전제로 연간 가이던스를 재확인.

<b>⚡ 서프라이즈 포인트</b>
- 매출 소폭 상회 vs EPS 소폭 하회: 매출 +0.9% beat, EPS -0.45% miss
- 북미 볼륨 기대 미달: PFNA 분기 중 볼륨 플랫. 가스 가격 부담과 일부 고객 가격 투자 실행 지연
- PBNA 마진 압박: 영업이익률 약 90bp 하락. 절반은 Alani 상업 계약, 나머지는 C-store/가스 채널·믹스

<b>🧾 특이사항 & 핵심 KPI</b>
- 상반기 보고 EPS +6%, 항시환율 EPS +3%
- 국제 볼륨 비중: 음료 전체의 2/3, 식품 50% 초과. 국제 매출 올해 $40bn 돌파 예상
- Q2 국제 영업이익률 +1%p
- 식품 permissible 포트폴리오 

## 7. Telegram preview

Build Telegram messages from the generated summaries before sending. Telegram has a message length limit, so long summaries are split automatically.


In [7]:
TELEGRAM_MAX_SUMMARIES = None  # Example: 1 for a single test message, None for all summaries
TELEGRAM_MESSAGE_LIMIT = 3900


def split_telegram_message(text: str, limit: int = TELEGRAM_MESSAGE_LIMIT) -> list[str]:
    text = text.strip()
    if len(text) <= limit:
        return [text]
    parts = []
    cursor = 0
    while cursor < len(text):
        end = min(cursor + limit, len(text))
        if end < len(text):
            split_at = text.rfind("\n\n", cursor, end)
            if split_at <= cursor + limit // 2:
                split_at = text.rfind("\n", cursor, end)
            if split_at <= cursor + limit // 2:
                split_at = end
            end = split_at
        parts.append(text[cursor:end].strip())
        cursor = end
    return [part for part in parts if part]


def sanitize_telegram_html(text: str) -> str:
    escaped = html.escape(text, quote=False)
    return escaped.replace("&lt;b&gt;", "<b>").replace("&lt;/b&gt;", "</b>")


def split_existing_telegram_title(body: str) -> tuple[str, str]:
    lines = body.strip().splitlines()
    if lines and lines[0].startswith("<b>") and lines[0].endswith("</b>"):
        title = lines[0].removeprefix("<b>").removesuffix("</b>").strip()
        rest = "\n".join(lines[1:]).strip()
        return title, rest
    return "", body.strip()


def build_telegram_messages(summary_results: list[dict], max_summaries: int | None = TELEGRAM_MAX_SUMMARIES) -> list[str]:
    selected_results = summary_results[:max_summaries] if max_summaries else summary_results
    messages = []
    for result in selected_results:
        body = result.get("summary", "").strip()
        title, body_without_title = split_existing_telegram_title(body)
        title = title or f"{result.get('ticker', 'UNKNOWN')} | {result.get('session', 'Unknown session')}"
        header = f"<b>{title}</b>"
        body = body_without_title
        full_message = f"{header}\n{body}".strip()
        if len(full_message) <= TELEGRAM_MESSAGE_LIMIT:
            messages.append(sanitize_telegram_html(full_message))
            continue
        body_limit = max(1000, TELEGRAM_MESSAGE_LIMIT - len(header) - 80)
        chunks = split_telegram_message(body, limit=body_limit)
        for part_idx, chunk in enumerate(chunks, start=1):
            messages.append(sanitize_telegram_html(f"{header}\nPart {part_idx}/{len(chunks)}\n\n{chunk}".strip()))
    return messages


telegram_messages = build_telegram_messages(results)
if not telegram_messages:
    raise RuntimeError("No Telegram messages were built. Run the summary cell first.")

print(f"Telegram messages ready: {len(telegram_messages)}")
print("\n--- Preview first message ---\n")
print(telegram_messages[0][:2000])


Telegram messages ready: 1

--- Preview first message ---

<b>PEP | FQ2 2026 Earnings Call</b>
<b>📊 핵심 실적 vs 컨센서스</b>
- EPS (Normalized): $2.20 vs 컨센 $2.21 (miss -0.45%)
- 매출: $24,181mm vs 컨센 $23,966.26mm (beat +0.90%)
- 통화: USD

<b>💬 경영진 핵심 메시지</b>
상반기 매출 약 7% 성장, 글로벌 볼륨은 식품 3%·음료 2%로 2022년 이후 최고 속도. 성장 축은 국제(7% 가속)이며 미국 식품은 볼륨·점유율 회복에 성공. 다만 북미는 가스 가격·impulse 채널 약세로 Q2 기대 미달. 하반기는 국제 강세 지속, 북미 점진 개선을 전제로 연간 가이던스를 재확인.

<b>⚡ 서프라이즈 포인트</b>
- 매출 소폭 상회 vs EPS 소폭 하회: 매출 +0.9% beat, EPS -0.45% miss
- 북미 볼륨 기대 미달: PFNA 분기 중 볼륨 플랫. 가스 가격 부담과 일부 고객 가격 투자 실행 지연
- PBNA 마진 압박: 영업이익률 약 90bp 하락. 절반은 Alani 상업 계약, 나머지는 C-store/가스 채널·믹스

<b>🧾 특이사항 &amp; 핵심 KPI</b>
- 상반기 보고 EPS +6%, 항시환율 EPS +3%
- 국제 볼륨 비중: 음료 전체의 2/3, 식품 50% 초과. 국제 매출 올해 $40bn 돌파 예상
- Q2 국제 영업이익률 +1%p
- 식품 permissible 포트폴리오 $3bn, 거의 두 자릿수 성장
- 미국 중장기 성장 가능 수준 약 3%로 언급

<b>🧭 가이던스 및 변경사항</b>
- 연간 가이던스: 재확인 (연초 고성장·EPS 목표 유지)
- H2 organic sales: 장기 목표 4~6% 하단 가시성 유지
- EPS: 범위 하단 쪽 가능성 언급
- 관세 환급: 연간 EPS 성장 약 1pt 기여, Q3 EPS 약 1pt 혜택 예상. 

## 8. Telegram connection check

Check whether the bot token works. If `TELEGRAM_CHAT_ID` is empty, send any message to the bot in Telegram and rerun this cell to inspect recent chat IDs.


In [8]:
TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN", "").strip()
TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID", "").strip()
TELEGRAM_API_BASE_URL = "https://api.telegram.org"


def telegram_api(method: str, payload: dict | None = None, timeout_sec: int = 60) -> dict:
    if not TELEGRAM_BOT_TOKEN:
        raise RuntimeError("Set TELEGRAM_BOT_TOKEN in .env first.")
    response = requests.post(
        f"{TELEGRAM_API_BASE_URL}/bot{TELEGRAM_BOT_TOKEN}/{method}",
        json=payload or {},
        timeout=timeout_sec,
    )
    if response.status_code >= 400:
        raise RuntimeError(f"Telegram API error {response.status_code}: {response.text}")
    data = response.json()
    if not data.get("ok"):
        raise RuntimeError(f"Telegram API returned ok=false: {data}")
    return data


bot_info = telegram_api("getMe")["result"]
print(f"Telegram bot connection OK: @{bot_info.get('username', 'unknown')}")

if TELEGRAM_CHAT_ID:
    print("TELEGRAM_CHAT_ID is set.")
else:
    updates = telegram_api("getUpdates").get("result", [])
    print("TELEGRAM_CHAT_ID is not set. Recent chat candidates:")
    for update in updates[-5:]:
        message = update.get("message") or update.get("channel_post") or {}
        chat = message.get("chat", {})
        if chat.get("id"):
            label = chat.get("title") or chat.get("username") or chat.get("first_name") or "unknown"
            print(f"- chat_id={chat.get('id')} label={label} type={chat.get('type')}")
    if not updates:
        print("No updates yet. Send a message to the bot in Telegram, then rerun this cell.")


Telegram bot connection OK: @RootN_Earnings_Alert_bot
TELEGRAM_CHAT_ID is set.


## 9. Telegram send

Set `TELEGRAM_BOT_TOKEN` and `TELEGRAM_CHAT_ID` in your environment. Keep `SEND_TELEGRAM = False` for dry-run checks, then change it to `True` when ready to send.


In [10]:
SEND_TELEGRAM = True
TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN", "").strip()
TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID", "").strip()
TELEGRAM_API_BASE_URL = "https://api.telegram.org"


def send_telegram_message(*, bot_token: str, chat_id: str, text: str, timeout_sec: int = 60) -> dict:
    response = requests.post(
        f"{TELEGRAM_API_BASE_URL}/bot{bot_token}/sendMessage",
        json={
            "chat_id": chat_id,
            "text": text,
            "parse_mode": "HTML",
            "disable_web_page_preview": True,
        },
        timeout=timeout_sec,
    )
    if response.status_code >= 400:
        raise RuntimeError(f"Telegram API error {response.status_code}: {response.text}")
    return response.json()


if not SEND_TELEGRAM:
    print("Dry run only. Set SEND_TELEGRAM = True to send messages.")
elif not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
    raise RuntimeError("Set TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID in your environment first.")
else:
    sent = []
    for idx, message in enumerate(telegram_messages, start=1):
        print(f"Sending Telegram message {idx}/{len(telegram_messages)}")
        sent.append(send_telegram_message(bot_token=TELEGRAM_BOT_TOKEN, chat_id=TELEGRAM_CHAT_ID, text=message))
        time.sleep(REQUEST_PAUSE_SEC)
    print(f"Sent Telegram messages: {len(sent)}")


Sending Telegram message 1/1
Sent Telegram messages: 1
